# Gemma 4 MT6985 Export (Google Colab)

This notebook bootstraps a Colab runtime to export `google/gemma-4-e2b-it` into an MT6985-targeted `.litertlm` bundle using the same helper scripts as this repo.

Before running:
- Use a High-RAM runtime.
- Add `HF_TOKEN` to Colab secrets or export it in the environment.
- Set `APP_REPO_URL` to the GitHub clone URL for the repo or branch that contains this notebook.
- If the notebook has not been merged to `main` yet, set `APP_REPO_REF` to the feature branch.

The notebook defaults to `CACHE_LENGTH=512` because that is the most realistic Colab-friendly starting point for the Gemma 4 export path.

In [ ]:
import os
import platform
import subprocess
from pathlib import Path

APP_REPO_URL = os.environ.get("APP_REPO_URL", "")
APP_REPO_REF = os.environ.get("APP_REPO_REF", "main")
LITERT_TORCH_REPO_URL = os.environ.get("LITERT_TORCH_REPO_URL", "https://github.com/google-ai-edge/litert-torch.git")
LITERT_TORCH_REF = os.environ.get("LITERT_TORCH_REF", "main")
MODEL_ID = os.environ.get("MODEL_ID", "google/gemma-4-e2b-it")
MODEL_FAMILY = "gemma4"
CACHE_LENGTH = int(os.environ.get("CACHE_LENGTH", "512"))
PREFILL_LENGTHS = os.environ.get("PREFILL_LENGTHS", "128")
QUANTIZATION_RECIPE = os.environ.get("QUANTIZATION_RECIPE", "dynamic_int4_block32")

workspace = Path("/content/mt6985-colab")
app_root = workspace / "app"
src_root = workspace / "src"
litert_torch_root = src_root / "litert-torch"
output_root = workspace / "output" / f"gemma4-mt6985-cache{CACHE_LENGTH}"
log_path = workspace / "output" / f"gemma4-mt6985-cache{CACHE_LENGTH}.log"

def run(command: str) -> None:
    print(f"$ {command}")
    subprocess.run(command, shell=True, check=True)

if platform.machine() != "x86_64":
    raise RuntimeError(f"Expected x86_64 runtime, got {platform.machine()}")

if not APP_REPO_URL:
    raise RuntimeError(
        "Set APP_REPO_URL to the GitHub clone URL for the repo or branch that contains this notebook before running it."
    )

workspace.mkdir(parents=True, exist_ok=True)
print(f"Workspace: {workspace}")
print(f"Output directory: {output_root}")

In [ ]:
try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    print("HF_TOKEN detected")
else:
    print("HF_TOKEN not set. Public access may still work, but authenticated access is recommended.")

In [ ]:
os.environ["PATH"] = f"{Path.home() / '.local' / 'bin'}:{os.environ['PATH']}"
run("apt-get update -y")
run("apt-get install -y git")
run("curl -LsSf https://astral.sh/uv/install.sh | sh")

if app_root.exists():
    run(f"rm -rf {app_root}")
if litert_torch_root.exists():
    run(f"rm -rf {litert_torch_root}")

src_root.mkdir(parents=True, exist_ok=True)
run(f"git clone --depth 1 --branch {APP_REPO_REF} {APP_REPO_URL} {app_root}")
run(f"git clone --depth 1 --branch {LITERT_TORCH_REF} {LITERT_TORCH_REPO_URL} {litert_torch_root}")

In [ ]:
inline_consts_path = litert_torch_root / "litert_torch" / "backend" / "inline_consts.py"
text = inline_consts_path.read_text()

if "import ml_dtypes" not in text:
    text = text.replace(
        "from litert_converter.mlir.dialects import arith\n",
        "from litert_converter.mlir.dialects import arith\nimport ml_dtypes\n",
        1,
    )

if "tensor.float().numpy().astype(ml_dtypes.bfloat16)" not in text:
    text = text.replace(
        "        arr = tensor.contiguous().detach().cpu().numpy()\n",
        "        tensor = tensor.contiguous().detach().cpu()\n        if tensor.dtype == torch.bfloat16:\n            arr = tensor.float().numpy().astype(ml_dtypes.bfloat16)\n        else:\n            arr = tensor.numpy()\n",
        1,
    )

if "def _tensor_to_mlir_compatible_resource_array" not in text:
    text = text.replace(
        "    return np.ascontiguousarray(arr)\n\n\ndef _get_tensor_uniform_value",
        "    return np.ascontiguousarray(arr)\n\n\ndef _tensor_to_mlir_compatible_resource_array(tensor: torch.Tensor) -> np.ndarray:\n    \"\"\"Converts a tensor to a lazily loaded numpy buffer for MLIR.\"\"\"\n    arr = _tensor_to_mlir_compatible_array(tensor)\n    if tensor.dtype == torch.bfloat16:\n        return arr.view(np.uint16)\n    return arr\n\n\ndef _get_tensor_uniform_value",
        1,
    )

text = text.replace(
    "    if x.dtype not in [torch.float32, torch.int32]:\n",
    "    if x.dtype not in [torch.float32, torch.int32, torch.bfloat16]:\n",
    1,
)
text = text.replace(
    "        arr = _tensor_to_mlir_compatible_array(x)\n        attr = ir.DenseResourceElementsAttr.get_from_buffer(\n",
    "        arr = _tensor_to_mlir_compatible_resource_array(x)\n        attr = ir.DenseResourceElementsAttr.get_from_buffer(\n",
    1,
)

for required_snippet in (
    "import ml_dtypes",
    "tensor.float().numpy().astype(ml_dtypes.bfloat16)",
    "def _tensor_to_mlir_compatible_resource_array",
    "torch.float32, torch.int32, torch.bfloat16",
    "_tensor_to_mlir_compatible_resource_array(x)",
):
    if required_snippet not in text:
        raise RuntimeError(f"Failed to patch inline_consts.py: missing {required_snippet}")

inline_consts_path.write_text(text)
print(f"Patched {inline_consts_path}")

In [ ]:
run(
    "bash -lc '"
    f"export PATH={Path.home() / '.local' / 'bin'}:$PATH && "
    f"cd {app_root} && "
    f"WORK_ROOT={app_root} SRC_ROOT={src_root} PYTHON_BIN=python3.12 ./tools/bootstrap_litert_export_env.sh"
    "'"
)

In [ ]:
output_root.parent.mkdir(parents=True, exist_ok=True)
run(
    "bash -lc '"
    f"export PATH={Path.home() / '.local' / 'bin'}:$PATH && "
    f"cd {app_root} && "
    f". .venv-litert/bin/activate && "
    f"PYTHONUNBUFFERED=1 python tools/build_mt6985_gemma3_litertlm.py --model {MODEL_ID} --model-family {MODEL_FAMILY} --cache-length {CACHE_LENGTH} --prefill-lengths {PREFILL_LENGTHS} --quantization-recipe {QUANTIZATION_RECIPE} --output-dir {output_root} --keep-intermediates 2>&1 | tee {log_path}"
    "'"
)

In [ ]:
run(f"find {output_root} -maxdepth 5 -type f | sort")
print(f"Log file: {log_path}")